# Usando a API do DataAn

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AknoLeonel/DataAn/blob/main/notebooks/api-dataan.ipynb)

O [DataAn](https://dataan.vercel.app/) publica todos os conjuntos como arquivos estáticos em JSON e
CSV: sem cadastro, sem chave, atualizados uma vez por dia. Este notebook mostra o caminho completo,
do catálogo a uma análise. Documentação: [dataan.vercel.app/api](https://dataan.vercel.app/api/).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import requests

BASE = "https://dataan.vercel.app/data"


def get(caminho: str) -> dict:
    resposta = requests.get(f"{BASE}/{caminho}", timeout=30)
    resposta.raise_for_status()
    return resposta.json()

## 1. O catálogo: o que existe e o quão atual está

In [ ]:
catalogo = get("catalog.json")
conjuntos = pd.DataFrame(catalogo["datasets"])
conjuntos[["slug", "title", "category", "frequency", "data_until", "stale"]]

## 2. Um conjunto completo

O JSON traz o que a página mostra: indicadores, gráficos com os pontos, análises em texto,
metodologia e fontes (com licença).

In [ ]:
ipca = get("datasets/ipca.json")
print(ipca["title"], "—", ipca["summary"])
pd.DataFrame(
    [{"indicador": k["label"], "valor": k["value"], "referência": k.get("as_of")} for k in ipca["kpis"]]
)

In [ ]:
for frase in ipca["insights"]:
    print("•", frase)
print()
for fonte in ipca["sources"]:
    print("Fonte:", fonte["name"], "—", fonte["url"], f"({fonte['license']})")

## 3. Os dados tabulares em CSV

In [ ]:
selic = pd.read_csv(f"{BASE}/csv/selic.csv", parse_dates=["mes"])
selic.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(selic["mes"], selic["selic_meta"], label="Selic (meta)")
ax.plot(selic["mes"], selic["ipca_12m"], label="IPCA 12 meses")
ax.fill_between(selic["mes"], selic["juro_real"], alpha=0.2, label="juro real ex-post")
ax.set_ylabel("% ao ano")
ax.legend()
ax.set_title("Selic, inflação e juro real")

## 4. Uma pergunta rápida: com que frequência o IPCA ficou acima do teto da meta?

In [ ]:
ipca_csv = pd.read_csv(f"{BASE}/csv/ipca.csv", parse_dates=["mes"])
acima = ipca_csv["ipca_12m"] > ipca_csv["meta_teto"]
por_ano = acima.groupby(ipca_csv["mes"].dt.year).mean().mul(100).round(0)
print(f"{acima.mean():.0%} dos meses desde {ipca_csv['mes'].dt.year.min()} ficaram acima do teto.")
por_ano.rename("% dos meses acima do teto").to_frame().T

## 5. Dengue: semanas, níveis de alerta e previsão

In [ ]:
dengue = pd.read_csv(f"{BASE}/csv/dengue-capitais.csv", parse_dates=["semana_inicio"])
ultimas = dengue[dengue["tipo"] == "observado"].sort_values("semana_inicio").groupby("cidade").tail(1)
ultimas[["cidade", "semana_inicio", "casos_estimados", "nivel_alerta", "rt"]]

## 6. Saúde do pipeline

`status.json` mostra se cada fonte respondeu na última execução. Se `stale` for verdadeiro, o arquivo
é a última versão válida e a coleta mais recente falhou.

In [ ]:
status = get("status.json")
print("Última execução:", status["generated_at"])
pd.DataFrame(status["datasets"])[["slug", "ok", "stale", "data_until", "rows"]]